## Notebook Overview
## Deep Kernel Learning PART 2

- This notebook evaluates how effectively **Deep Kernel Learning (DKL)** models can predict **ISUP prostate cancer grade** using **radiomics features** extracted from prostate MRI. It follows the same structure and methodology as **02‑deep_kernel_PART1.ipynb** but here the focus is on exploring the hyperparameter space using insights gained from the previous notebook.

Parts of the code were adapted and modified from the GPyTorch documentation: https://docs.gpytorch.ai/en/stable/examples/06_PyTorch_NN_Integration_DKL/KISSGP_Deep_Kernel_Regression_CUDA.html

In [1]:

import tqdm
import torch
import gpytorch
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from deep_gp.preprocessing_data import load_data, undersample_class0, apply_smote
from deep_gp.deep_kernel_class import GPRegressionModel
from tqdm import tqdm


import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))


%matplotlib inline
%load_ext autoreload
%autoreload 2



In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
data = load_data()
df_new = undersample_class0(data)
df_resampled = apply_smote(df_new)

X_balanced = df_resampled.drop(columns=["case_ISUP"])
y_balanced = df_resampled["case_ISUP"]

all_features = df_resampled.drop(columns=["case_ISUP"]).columns.tolist()


In [4]:
def run_dkl_experiment(feature_list, df,
                       latent_dim=10,
                       extractor_type="large",
                       activation="relu",
                       kernel_type="rbf_ard",
                       noise_value=0.05,
                       learning_rate=0.01,
                       n_epochs=300):

    print(
    f"\nRunning DKL with {len(feature_list)} features, "
    f"latent_dim={latent_dim}, extractor={extractor_type}, "
    f"activation={activation}, kernel={kernel_type}, "
    f"noise={noise_value}, lr={learning_rate}"
)

    X = df[feature_list]
    y = df["case_ISUP"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1)).ravel()
    y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1)).ravel()
    
    train_x = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    train_y = torch.tensor(y_train_scaled, dtype=torch.float32).to(device)
    test_x  = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    test_y  = torch.tensor(y_test_scaled, dtype=torch.float32).to(device)

    
    likelihood = gpytorch.likelihoods.GaussianLikelihood()
    # Case 1: homoskedastic GP (learn noise automatically)
    if noise_value == "learned":
        model = GPRegressionModel(
            train_x, train_y, likelihood,
            data_dim=train_x.shape[1],
            latent_dim=latent_dim,
            extractor_type=extractor_type,
            activation=activation,
            kernel_type=kernel_type,
            noise_value=None   # tell the model not to override noise
        )

    # Case 2: fixed-noise GP (heteroskedastic search)
    else:
        model = GPRegressionModel(
            train_x, train_y, likelihood,
            data_dim=train_x.shape[1],
            latent_dim=latent_dim,
            extractor_type=extractor_type,
            activation=activation,
            kernel_type=kernel_type,
            noise_value=noise_value
        )
    model = model.to(device)
    likelihood = likelihood.to(device)



    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    model.train()
    likelihood.train()

    for i in range(n_epochs):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        optimizer.step()

    model.eval()
    likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = likelihood(model(test_x))

    # Predictions
    y_pred = scaler_y.inverse_transform(preds.mean.cpu().numpy().reshape(-1,1)).ravel()
    y_true = scaler_y.inverse_transform(test_y.cpu().numpy().reshape(-1,1)).ravel()

    # compute predictive uncertainty
    f_std = preds.stddev.cpu().numpy().ravel()
    f_std = f_std * scaler_y.scale_[0]   # convert to ISUP scale
    
    # Build uncertainty dataframe
    df_unc = pd.DataFrame({
        "true": y_true,
        "pred": y_pred,
        "std": f_std
    })
    df_unc["true_class"] = np.round(df_unc["true"]).astype(int)

    # Metrics
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)

    print(f"MSE={mse:.4f}, R²={r2:.4f}")

    return mse, r2, df_unc


In [5]:
feature_sets = {
    "all": all_features
}

extractors = ["small", "medium"]
activations = ["relu", "tanh"]
latent_dims = [15, 20]
kernels = ["matern_15", "matern_25","rbf_ard"]
noise_values = ["learned", 0.01, 0.05, 0.1]
learning_rates = [0.01, 0.005]

results = []

# Count total runs for tqdm
total_runs = (
    len(feature_sets)
    * len(extractors)
    * len(activations)
    * len(latent_dims)
    * len(kernels)
    * len(noise_values)
    * len(learning_rates)
)

pbar = tqdm(total=total_runs, desc="DKL Hyperparameter Search", ncols=100)

for feat_name, feat_list in feature_sets.items():
    print(f"\n==============================")
    print(f"   Testing feature set: {feat_name}")
    print(f"==============================")

    for ext in extractors:
        for act in activations:
            for ld in latent_dims:
                for kernel in kernels:
                    for noise in noise_values:
                        for lr in learning_rates:

                            pbar.set_postfix({
                                "feat": feat_name,
                                "ext": ext,
                                "ld": ld,
                                "act": act,
                                "kernel": kernel,
                                "noise": noise,
                                "lr": lr
                            })

                            mse, r2, df_unc = run_dkl_experiment(
                                feature_list=feat_list,
                                df=df_resampled,
                                latent_dim=ld,
                                extractor_type=ext,
                                activation=act,
                                kernel_type=kernel,
                                noise_value=noise,
                                learning_rate=lr
                            )

                            results.append({
                                "features": feat_name,
                                "extractor": ext,
                                "activation": act,
                                "latent_dim": ld,
                                "kernel": kernel,
                                "noise": noise,
                                "lr": lr,
                                "mse": mse,
                                "r2": r2,
                                "uncertainty": df_unc
                            })

                            pbar.update(1)

pbar.close()


DKL Hyperparameter Search:   0%| | 0/192 [00:00<?, ?it/s, feat=all, ext=small, ld=15, act=relu, kern


   Testing feature set: all

Running DKL with 107 features, latent_dim=15, extractor=small, activation=relu, kernel=matern_15, noise=learned, lr=0.01


DKL Hyperparameter Search:   1%| | 1/192 [00:03<12:17,  3.86s/it, feat=all, ext=small, ld=15, act=re

MSE=1.5735, R²=0.5068

Running DKL with 107 features, latent_dim=15, extractor=small, activation=relu, kernel=matern_15, noise=learned, lr=0.005


DKL Hyperparameter Search:   1%| | 2/192 [00:06<10:00,  3.16s/it, feat=all, ext=small, ld=15, act=re

...

MSE=1.8748, R²=0.4123

Running DKL with 107 features, latent_dim=20, extractor=medium, activation=tanh, kernel=rbf_ard, noise=0.1, lr=0.01


DKL Hyperparameter Search:  99%|▉| 191/192 [09:00<00:02,  2.67s/it, feat=all, ext=medium, ld=20, act

MSE=2.1999, R²=0.3104

Running DKL with 107 features, latent_dim=20, extractor=medium, activation=tanh, kernel=rbf_ard, noise=0.1, lr=0.005


DKL Hyperparameter Search: 100%|█| 192/192 [09:02<00:00,  2.83s/it, feat=all, ext=medium, ld=20, act

MSE=1.9695, R²=0.3826


In [6]:
results_sorted = sorted(results, key=lambda x: x["r2"], reverse=True)
best = results_sorted[0]

print("\n===== BEST MODEL FOUND =====")
print(f"Feature set:   {best['features']}")
print(f"Extractor:     {best['extractor']}")
print(f"Activation:    {best['activation']}")
print(f"Latent dim:    {best['latent_dim']}")
print(f"Noise:         {best['noise']}")
print(f"Kernel:        {best['kernel']}")
print(f"LR:            {best['lr']}")
print(f"MSE:           {best['mse']:.4f}")
print(f"R²:            {best['r2']:.4f}")

df_unc_best = best["uncertainty"]

unc_by_class = df_unc_best.groupby("true_class")["std"].mean()

print("\n===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====")
for cls, std in unc_by_class.items():
    print(f"ISUP {cls}:  std = {std:.4f}")



===== BEST MODEL FOUND =====
Feature set:   all
Extractor:     small
Activation:    relu
Latent dim:    20
Noise:         0.01
Kernel:        matern_25
LR:            0.005
MSE:           1.2936
R²:            0.5945

===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====
ISUP 0:  std = 0.7670
ISUP 1:  std = 0.7713
ISUP 2:  std = 0.7703
ISUP 3:  std = 0.6354
ISUP 4:  std = 0.4987
ISUP 5:  std = 0.5353
